## 批量处理所有样本
对每个样本进行质控，针对scRNA-seq数据，质控标准如下：200–10,000 genes, 8,000–80,000 gene counts, and less than 10% mitochondrial gene counts；去除被Scrublet预测为doublets

针对scATAC-seq数据，质控标准如下：1,000-20，000 fragments peaks，greater than 15% reads peaks， less than 5% ENCODE blacklist regions, less than 5 nucleosome banding pattern, and greater than enrichment-score for Tn5-integration events at transcriptional start sites

In [1]:
library(Seurat)
library(ggplot2)
library(patchwork)
library(dplyr)
library(Signac)
library(GenomicRanges)
library(GenomeInfoDb)
library(EnsDb.Hsapiens.v86)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following object is masked from ‘package:SeuratObject’:

    intersect


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, 

In [9]:
annotation <- GetGRangesFromEnsDb(ensdb = EnsDb.Hsapiens.v86)
seqlevels(annotation) <- paste0("chr", seqlevels(annotation))

Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warn

In [ ]:
files <- list.files("/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/0_raw_data")
files

In [15]:
QC_number<-list()

批量处理所有的癌症样本，但需注意可能其中存在由于doublet之后数量很少的样本导致报错，可以选择更改代码的循环起止再次继续跑代码，即`print(i)`后代码报错，从`i+2`次重新运行

In [ ]:
#批量处理所有的癌症样本，但需注意可能其中存在由于doublet之后数量很少的样本导致报错，可以选择更改代码的
for(i in 1:length(files)){
    counts <- Read10X(data.dir = paste0("/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/0_raw_data/",files[i]))
    fragpath<-paste0("/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/0_raw_data/",files[i],"/",files[i],"-atac_fragments.tsv.gz")
    scrublet_res <- read.csv(paste0("/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/1_processed_data/",files[i],"-_Scrublet_Results.csv"), row.names = 1)
    seurat_obj <-CreateSeuratObject(
        counts = counts$`Gene Expression`,
        assay = "RNA",
        project = files[i]
    )
    seurat_obj[["ATAC"]] <- CreateChromatinAssay(counts = counts$Peaks,
                                             sep = c(":", "-"),
                                             fragments = fragpath,
                                             annotation = annotation)
                             
    seurat_obj <- AddMetaData(seurat_obj, metadata = scrublet_res) 
    print(seurat_obj)
    seurat_clean <- subset(x = seurat_obj,subset = !is.na(predicted_doublet_final) & (predicted_doublet_final == "False" | predicted_doublet_final == FALSE))
    saveRDS(seurat_clean, file = paste0("/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/1_processed_data/",files[i],"-_filt_doublets.rds"))
    print(seurat_clean)
    seurat_clean@assays[["ATAC"]]@fragments[[1]]@path=fragpath
    seurat_clean$percent.mt <- PercentageFeatureSet(seurat_clean, pattern = "^MT-", assay = "RNA")
    DefaultAssay(seurat_clean) <- "ATAC"
    seurat_clean <- NucleosomeSignal(seurat_clean)
    seurat_clean <- TSSEnrichment(seurat_clean)
    seurat_clean$blacklist_ratio <- FractionCountsInRegion(
        object = seurat_clean, 
        assay = 'ATAC',
        regions = blacklist_hg38_unified # GRanges对象，定义了ENCODE hg38黑名单区域
    )
   seurat_final <- subset(seurat_clean, subset = 
                         nFeature_RNA >=200 & 
                          nFeature_RNA <= 10000 & 
                          nCount_RNA >= 1000 & 
                          nCount_RNA <= 80000 & 
                          percent.mt <= 10 &
                          nCount_ATAC > 1000 &
                          nCount_ATAC < 20000 &
                          blacklist_ratio < 0.05 &
                          nucleosome_signal < 5 &
                          TSS.enrichment > 2
                         )
    print(seurat_final)
    peaks <- CallPeaks(seurat_final)
    peaks <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
    peaks <- subsetByOverlaps(x = peaks, ranges = blacklist_hg38_unified, invert = TRUE)
    macs2_counts <- FeatureMatrix(fragments = Fragments(seurat_final),features = peaks,cells = colnames(seurat_final))
    seurat_final[["peaks"]] <- CreateChromatinAssay(counts = macs2_counts,fragments = fragpath,annotation = annotation)
    print(seurat_final)
    DefaultAssay(seurat_final) <- "RNA"
    seurat_final <- SCTransform(seurat_final,vars.to.regress = c("nCount_RNA", "percent.mt"), return.only.var.genes = F)
    seurat_final <- RunPCA(seurat_final)
    DefaultAssay(seurat_final) <- "peaks"
    seurat_final <- FindTopFeatures(seurat_final)
    seurat_final <- RunTFIDF(seurat_final)
    seurat_final <- RunSVD(seurat_final)
    seurat_final <- FindMultiModalNeighbors(object = seurat_final,reduction.list = list("pca", "lsi"), 
                                      dims.list = list(1:30, 2:30),modality.weight.name = "RNA.weight",verbose = TRUE)
    seurat_final <- RunUMAP(object = seurat_final,nn.name = "weighted.nn",assay = "RNA",verbose = TRUE)
    DefaultAssay(seurat_final) <- "SCT"
    seurat_final <- FindNeighbors(seurat_final, dims = 1:30)
    seurat_final <- FindClusters(seurat_final, resolution = 0.5, verbose = FALSE,algorithm = 3)
    saveRDS(seurat_final, file = paste0("/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/2_seurat_filt/",files[i],".rds"))
    QC_number[[files[i]]]<-data.frame(raw= dim(seurat_obj)[2],QC1=dim(scrublet_res)[1],Scrublet=dim(seurat_clean)[2], QC2=dim(seurat_final)[2]) 
    print(i)
}

10X data contains more than one type and is being returned as a list containing matrices of each type.

Computing hash



An object of class Seurat 
161569 features across 625129 samples within 2 assays 
Active assay: RNA (36601 features, 0 variable features)
 1 layer present: counts
 1 other assay present: ATAC


Warning message:
"Removing 621863 cells missing data for vars requested"


An object of class Seurat 
161569 features across 2855 samples within 2 assays 
Active assay: RNA (36601 features, 0 variable features)
 1 layer present: counts
 1 other assay present: ATAC


Extracting TSS positions

Extracting fragments at TSSs


Computing TSS enrichment score



...

An object of class Seurat 
266235 features across 1513 samples within 3 assays 
Active assay: ATAC (104359 features, 0 variable features)
 2 layers present: counts, data
 2 other assays present: RNA, peaks


Running SCTransform on assay: RNA

vst.flavor='v2' set. Using model with fixed slope and excluding poisson genes.

Calculating cell attributes from input UMI matrix: log_umi

Variance stabilizing transformation of count matrix of size 16445 by 1513

Model formula is y ~ log_umi

Get Negative Binomial regression parameters per gene

Using 2000 genes, 1513 cells

Found 64 outliers - those will be ignored in fitting/regularization step


Second step: Get residuals using fitted parameters for 16445 genes

Computing corrected count matrix for 16445 genes

Calculating gene attributes

Wall clock passed: Time difference of 13.93829 secs

Determine variable features

Regressing out nCount_RNA, percent.mt

Centering data matrix

Place corrected count matrix in counts slot

Set default assay to SCT

PC_ 1 
Positive:  ERO1A, MDM2, NDRG1, NEBL, PDLIM5, MUC16, LINC02755, FAM214A, HIF1A-AS3, NPEPPS 
	   NT5C2, ABTB2, MACC1, TBC1D8, GPCPD1, NEAT1, MBOAT2, MXD1, KDM7A, MUC4 
	   MYO1E, DUSP10, MYO5B, 

[1] 3


10X data contains more than one type and is being returned as a list containing matrices of each type.

Computing hash



ERROR: Error in CreateFragmentObject(path = fragments, cells = cells, validate.fragments = validate.fragments, : Not all cells requested could be found in the fragment file.


In [32]:
save(QC_number,file="/mnt/netshare2/liangxuan/data/cancer/QC.rda")

In [38]:
QC_number1 <- Reduce(rbind, QC_number)

In [40]:
dim(QC_number1)

[1] 133   5

In [44]:
rownames(QC_number1)<-files[!(files%in%c("CM268C1-T1","P5296-1N1","P5576-1N1"))]

In [45]:
head(QC_number1)

,raw1,raw2,QC1,Scrublet,QC2
,<int>,<dbl>,<int>,<int>,<int>
CE336E1-S1,625129,625129,3266,2855,654
CE337E1-S1,696174,696174,27914,22523,861
CE338E1-S1,684905,684905,22168,18909,1513
CE339E1-S1,646622,528212,37618,31442,2009
CE340E1-S1,700480,700480,24652,20384,7035
CE342E1-S1,700911,700911,52529,44074,2587


In [47]:
write.csv(QC_number1,file="/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/QC_cellnember.csv",quote=F)